[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch10-model-compression/01-cat-tia-va-do-thua.ipynb)

# Cắt tỉa: số 0 nào làm mô hình nhỏ lại, số 0 nào làm nó nhanh hơn

Chương 10 nói một câu mà sổ tay này sẽ kiểm tra bằng số đo: **số 0 chỉ có giá trị khi đường
thực thi bỏ qua được chúng**. Chúng ta sẽ cắt tỉa một mạng nhỏ đã huấn luyện trên MNIST, xem
độ chính xác còn lại bao nhiêu, rồi hỏi hai câu mà độ chính xác không trả lời: phép nhân có
nhanh hơn không, và tệp mô hình có nhỏ hơn không.

**Bạn sẽ làm:**
1. cắt tỉa theo độ lớn trên một ma trận 3 nhân 3, đúng ý của listing 1;
2. huấn luyện một perceptron nhiều tầng (multilayer perceptron) trên 10,000 ảnh MNIST;
3. so cắt tỉa toàn cục với cắt tỉa theo từng tầng, ở sáu mức độ thưa;
4. so cắt tỉa không cấu trúc với cắt tỉa có cấu trúc ở cùng số trọng số;
5. đo thời gian của phép nhân ma trận (matrix multiplication) dày, dày có số 0, dày nhỏ hơn, và thưa;
6. đếm byte của bốn cách lưu một ma trận thưa, rồi tìm điểm hoà vốn.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

## 1. Cắt tỉa theo độ lớn, trên một ma trận nhỏ

Bài toán cắt tỉa của chương là cực tiểu hoá hàm mất mát (loss function) với ràng buộc
$\|\hat{\mathbf W}\|_0 \le k$: số trọng số khác 0 không quá $k$. Bài toán ấy mang tính tổ hợp,
nên thực tế dùng một quy tắc kinh nghiệm: giữ $k$ trọng số có giá trị tuyệt đối lớn nhất, đặt
phần còn lại về 0. Listing 1 của chương đưa số tham số khác 0 từ 9 xuống 4. Ma trận dưới đây là
của sổ tay, còn $k$ là của sách.

In [ ]:
W = torch.tensor([[ 0.82, -0.05,  0.31],
                  [-0.02,  0.67, -0.44],
                  [ 0.09, -0.91,  0.03]])
k = sach(4, nguon="k = 4")

nguong = W.abs().flatten().sort(descending=True).values[k - 1]  # độ lớn nhỏ nhất còn được giữ
mat_na = (W.abs() >= nguong).float()                             # mặt nạ nhị phân: 1 giữ, 0 bỏ
W_thua = W * mat_na

print("mặt nạ:\n", mat_na)
print("ma trận sau khi cắt:\n", W_thua)
theo_sach("số tham số khác 0 trước khi cắt", int((W != 0).sum()), sach=9, nguon="9 → 4")
theo_sach("số tham số khác 0 sau khi cắt", int((W_thua != 0).sum()), sach=4, nguon="9 → 4 · k = 4")

Lưu ý: `W_thua` vẫn là một tensor 3 nhân 3, đủ 9 ô, 36 byte ở FP32. Số 0 đã có mặt, nhưng chưa
có gì nhỏ đi. Phần 6 sẽ quay lại đúng điểm này.

## 2. Một mạng để cắt

Mạng có ba tầng kết nối đầy đủ (fully connected layer), 784-256-128-10, học 5 epoch trên
10,000 ảnh huấn luyện. Độ chính xác đo trên cả 10,000 ảnh kiểm tra. Mỗi lần chạy lại cho cùng
kết quả trên cùng máy, vì hạt giống đã được đặt.

In [ ]:
import copy

import torch.nn.functional as F
from torchvision import datasets


def nap_mnist(train, n):
    ds = datasets.MNIST(DATA, train=train, download=True)
    x = ds.data[:n].float().div(255).view(-1, 784)
    return (x - 0.1307) / 0.3081, ds.targets[:n]   # chuẩn hoá theo trung bình và độ lệch của MNIST


X, y = nap_mnist(True, 10_000)
X_kt, y_kt = nap_mnist(False, 10_000)


def tao_mlp():
    return torch.nn.Sequential(torch.nn.Linear(784, 256), torch.nn.ReLU(),
                               torch.nn.Linear(256, 128), torch.nn.ReLU(),
                               torch.nn.Linear(128, 10))


def cac_tang(m):
    return [l for l in m if isinstance(l, torch.nn.Linear)]


def huan_luyen(m, so_epoch, mat_na=None, lr=1e-3):
    """Adam, batch 64. Nếu có mặt nạ, trọng số đã cắt được giữ ở 0 sau mỗi bước."""
    opt = torch.optim.Adam(m.parameters(), lr)
    for _ in range(so_epoch):
        thu_tu = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            chon = thu_tu[i:i + 64]
            loss = F.cross_entropy(m(X[chon]), y[chon])
            opt.zero_grad()
            loss.backward()
            opt.step()
            if mat_na is not None:
                with torch.no_grad():
                    for l, mk in zip(cac_tang(m), mat_na):
                        l.weight.mul_(mk)
    return m


def do_chinh_xac(m):
    with torch.no_grad():
        return (m(X_kt).argmax(1) == y_kt).float().mean().item()


dat_hat_giong()
t0 = time.perf_counter()
goc = huan_luyen(tao_mlp(), 5)
do_tren_may("thời gian huấn luyện 5 epoch", time.perf_counter() - t0, "s")
acc_goc = do_chinh_xac(goc)
do_tren_may("độ chính xác của mạng gốc", 100 * acc_goc, "%")

so_trong_so = sum(l.weight.numel() for l in cac_tang(goc))
print(f"số trọng số (không kể độ lệch): {so_trong_so:,}")
assert so_trong_so == 784 * 256 + 256 * 128 + 128 * 10

## 3. Cắt toàn cục hay cắt theo từng tầng

Có hai cách chọn ngưỡng cho cùng một độ thưa, ví dụ 90 phần trăm:

* **theo từng tầng**: mỗi tầng tự bỏ 90 phần trăm trọng số nhỏ nhất của nó;
* **toàn cục**: xếp hạng mọi trọng số của cả mạng chung một bảng, rồi bỏ 90 phần trăm nhỏ nhất.

Chương nhắc rằng thang đo thô giữa các tầng có thể khác nhau. Cách toàn cục để thang đo ấy tự
quyết tầng nào mất nhiều hơn.

### Dự đoán

Không tinh chỉnh lại gì cả, ở độ thưa 90 phần trăm, cách nào giữ độ chính xác tốt hơn? Và theo
bạn, cách toàn cục sẽ cắt tầng nào mạnh nhất: tầng đầu 784 nhân 256, hay tầng ra 128 nhân 10?

In [ ]:
def mat_na_toan_cuc(m, do_thua):
    w = torch.cat([l.weight.detach().abs().flatten() for l in cac_tang(m)])
    nguong = w.kthvalue(int(round(do_thua * w.numel()))).values
    return [(l.weight.detach().abs() > nguong).float() for l in cac_tang(m)]


def mat_na_tung_tang(m, do_thua):
    out = []
    for l in cac_tang(m):
        w = l.weight.detach().abs().flatten()
        nguong = w.kthvalue(int(round(do_thua * w.numel()))).values
        out.append((l.weight.detach().abs() > nguong).float())
    return out


def ap_mat_na(m, mat_na):
    m = copy.deepcopy(m)
    with torch.no_grad():
        for l, mk in zip(cac_tang(m), mat_na):
            l.weight.mul_(mk)
    return m


cac_do_thua = [0.5, 0.7, 0.8, 0.9, 0.95, 0.98]
acc_toan_cuc, acc_tung_tang, mat_do_tang = [], [], []
for s in cac_do_thua:
    mk = mat_na_toan_cuc(goc, s)
    acc_toan_cuc.append(do_chinh_xac(ap_mat_na(goc, mk)))
    mat_do_tang.append([m.mean().item() for m in mk])
    acc_tung_tang.append(do_chinh_xac(ap_mat_na(goc, mat_na_tung_tang(goc, s))))
    # cùng một độ thưa tổng: đếm lại cho chắc
    giu = sum(int(m.sum()) for m in mk)
    assert abs(giu / so_trong_so - (1 - s)) < 1e-3

for s, a, b in zip(cac_do_thua, acc_toan_cuc, acc_tung_tang):
    print(f"độ thưa {s:.0%}: toàn cục {a:.2%} · theo từng tầng {b:.2%}")

fig, ax = plt.subplots()
ax.plot([100 * s for s in cac_do_thua], [100 * a for a in acc_toan_cuc], "o-", label="toàn cục")
ax.plot([100 * s for s in cac_do_thua], [100 * a for a in acc_tung_tang], "s-", label="theo từng tầng")
ax.axhline(100 * acc_goc, color="0.5", lw=1, ls="--")
ax.annotate("mạng gốc", (50, 100 * acc_goc), xytext=(0, -14), textcoords="offset points", color="0.4")
ax.set_xlabel("độ thưa (phần trăm trọng số bị cắt)")
ax.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax.set_title("Cắt tỉa theo độ lớn, chưa tinh chỉnh lại")
ax.set_ylim(0, 100)
ax.legend(loc="lower left")
plt.show()

### Cắt toàn cục để lại tầng nào dày hơn?

Đây là kiểu chẩn đoán bằng hình ảnh mà chương nói tới ở hình 32: độ thưa dồn vào đâu. Biểu đồ
dưới cho tỉ lệ trọng số còn lại của từng tầng khi cả mạng bị cắt toàn cục.

In [ ]:
ten_tang = ["784→256", "256→128", "128→10"]
fig, ax = plt.subplots(figsize=(7, 3.6))
rong = 0.13
for j, s in enumerate(cac_do_thua):
    ax.bar(np.arange(3) + (j - 2.5) * rong, [100 * d for d in mat_do_tang[j]], rong,
           color=plt.cm.Blues(0.35 + 0.1 * j), label=f"{s:.0%}")
ax.set_xticks(range(3), ten_tang)
ax.set_xlabel("tầng")
ax.set_ylabel("trọng số còn lại (%)")
ax.set_title("Cắt toàn cục: mỗi tầng giữ lại bao nhiêu")
ax.legend(title="độ thưa toàn mạng", loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
plt.tight_layout()
plt.show()

# Tầng đầu: mỗi điểm ảnh còn giữ bao nhiêu trọng số, ở độ thưa 90 phần trăm
mk90 = mat_na_toan_cuc(goc, 0.9)[0]          # 256 x 784
fig, ax = plt.subplots(figsize=(4.2, 3.6))
anh = ax.imshow(mk90.sum(0).view(28, 28), cmap="Blues")
ax.set_title("Tầng đầu, độ thưa 90 %:\nsố trọng số còn lại của mỗi điểm ảnh", fontsize=10)
ax.axis("off")
fig.colorbar(anh, ax=ax, fraction=0.046)
plt.show()

Cắt toàn cục cắt tầng đầu mạnh nhất, vì tầng ấy có nhiều trọng số nhất và phần lớn chúng nhỏ.
Tầng ra nhỏ, mỗi trọng số của nó gánh nhiều hơn, nên nó giữ lại nhiều nhất. Cắt theo từng tầng
bắt tầng ra mất cùng tỉ lệ như tầng đầu, và ở độ thưa cao đó là chỗ mạng gãy trước. Hình điểm
ảnh cho thấy thêm một điều: trọng số nối vào phần viền của ảnh, nơi chữ số hầu như không bao giờ
chạm tới, là những thứ bị bỏ trước.

## 4. Tinh chỉnh để phục hồi

Chương nói cắt tỉa thường xen với tinh chỉnh, để phần trọng số còn lại thích nghi. Ở đây ta cắt
toàn cục 95 phần trăm rồi tinh chỉnh một epoch, giữ trọng số đã cắt ở đúng 0.

Một phép so công bằng cần một đối chứng: mạng gốc cũng được học thêm một epoch. Nếu không, phần
tăng có thể chỉ là nhờ được huấn luyện lâu hơn.

In [ ]:
dat_hat_giong(1)
mk95 = mat_na_toan_cuc(goc, 0.95)
cat95 = ap_mat_na(goc, mk95)
acc_truoc = do_chinh_xac(cat95)
huan_luyen(cat95, 1, mat_na=mk95)
acc_sau = do_chinh_xac(cat95)
assert all(int((l.weight[mk == 0] != 0).sum()) == 0 for l, mk in zip(cac_tang(cat95), mk95))

dat_hat_giong(1)
doi_chung = huan_luyen(copy.deepcopy(goc), 1)
do_tren_may("cắt 95 %, chưa tinh chỉnh", 100 * acc_truoc, "%")
do_tren_may("cắt 95 %, sau 1 epoch tinh chỉnh", 100 * acc_sau, "%")
do_tren_may("đối chứng: mạng dày học thêm 1 epoch", 100 * do_chinh_xac(doi_chung), "%")

## 5. Không cấu trúc hay có cấu trúc, ở cùng số trọng số

Cắt tỉa có cấu trúc bỏ cả nơ-ron (neuron): một hàng của ma trận trọng số tầng trước và một cột
của tầng sau. Ở đây mỗi nơ-ron ẩn được chấm điểm bằng chuẩn L2 của các trọng số đi vào nó, và
những nơ-ron điểm thấp bị bỏ. Kết quả là một mạng **dày nhỏ hơn**, chạy bằng đúng phép nhân ma
trận thông thường.

Mỗi cấu hình có cấu trúc giữ một tỉ lệ trọng số. Ta cắt không cấu trúc (toàn cục) tới đúng tỉ lệ
ấy, rồi tinh chỉnh cả hai một epoch.

### Dự đoán

Ở cùng số trọng số còn lại, bên nào giữ độ chính xác tốt hơn? Và bên nào bạn nghĩ sẽ chạy
nhanh hơn trên một CPU bình thường?

In [ ]:
def cat_noron(m, giu1, giu2):
    """Giữ giu1 nơ-ron ở tầng ẩn thứ nhất và giu2 ở tầng ẩn thứ hai, theo chuẩn L2 trọng số vào."""
    L = cac_tang(m)
    i1 = L[0].weight.norm(dim=1).argsort(descending=True)[:giu1].sort().values
    i2 = L[1].weight[:, i1].norm(dim=1).argsort(descending=True)[:giu2].sort().values
    moi = torch.nn.Sequential(torch.nn.Linear(784, giu1), torch.nn.ReLU(),
                              torch.nn.Linear(giu1, giu2), torch.nn.ReLU(),
                              torch.nn.Linear(giu2, 10))
    A, B, C = cac_tang(moi)
    with torch.no_grad():
        A.weight.copy_(L[0].weight[i1]); A.bias.copy_(L[0].bias[i1])
        B.weight.copy_(L[1].weight[i2][:, i1]); B.bias.copy_(L[1].bias[i2])
        C.weight.copy_(L[2].weight[:, i2]); C.bias.copy_(L[2].bias)
    return moi


cau_hinh = [(128, 64), (64, 32), (32, 16)]
ti_le, acc_ct, acc_kct = [], [], []
for g1, g2 in cau_hinh:
    dat_hat_giong(2)
    ct = huan_luyen(cat_noron(goc, g1, g2), 1)
    f = sum(l.weight.numel() for l in cac_tang(ct)) / so_trong_so
    dat_hat_giong(2)
    mk = mat_na_toan_cuc(goc, 1 - f)
    kct = huan_luyen(ap_mat_na(goc, mk), 1, mat_na=mk)
    ti_le.append(f)
    acc_ct.append(do_chinh_xac(ct))
    acc_kct.append(do_chinh_xac(kct))
    print(f"giữ {f:6.1%} trọng số: có cấu trúc (784-{g1}-{g2}-10) {acc_ct[-1]:.2%}"
          f" · không cấu trúc {acc_kct[-1]:.2%}")

fig, ax = plt.subplots()
ax.plot([100 * f for f in ti_le], [100 * a for a in acc_kct], "o-", label="không cấu trúc (mặt nạ)")
ax.plot([100 * f for f in ti_le], [100 * a for a in acc_ct], "s-", label="có cấu trúc (bỏ nơ-ron)")
for f, (g1, g2), a in zip(ti_le, cau_hinh, acc_ct):
    ax.annotate(f"784-{g1}-{g2}-10", (100 * f, 100 * a), xytext=(-8, 10), textcoords="offset points",
                fontsize=8, color="0.3", ha="right")
ax.set_xscale("log")
ax.set_xlim(8, 60)
ax.set_xticks([100 * f for f in ti_le], [f"{100 * f:.0f}" for f in ti_le])
ax.minorticks_off()
ax.set_xlabel("trọng số còn lại (%, thang log)")
ax.set_ylabel("độ chính xác sau 1 epoch tinh chỉnh (%)")
ax.set_title("Cùng số trọng số, hai cách cắt")
ax.legend(loc="lower right")
plt.show()

Cắt không cấu trúc linh hoạt hơn: nó được chọn từng trọng số, nên thường giữ độ chính xác tốt
hơn ở cùng số trọng số. Nhưng nó để lại những ma trận kích thước cũ đầy số 0 rải rác. Cắt có
cấu trúc để lại ma trận nhỏ hơn thật. Phần tiếp theo đo xem điều đó đổi gì về thời gian.

## 6. Số 0 có làm phép nhân nhanh hơn không

Chương viết rằng một mô hình thưa với 90 phần trăm trọng số bằng 0 vẫn có thể chạy gần đủ chi
phí trên phần cứng không thiết kế cho truy cập bất thường. Bộ xử lý không bỏ qua được một phép
nhân trừ khi biết toán hạng bằng 0, mà muốn biết thì phải nạp nó trước.

Ta đo bốn cách tính cùng một tầng 2048 nhân 2048 trên một batch 64 cột:

1. ma trận dày, không có số 0;
2. cùng ma trận, 90 phần trăm phần tử đã đặt về 0 nhưng vẫn lưu dày;
3. ma trận dày nhỏ hơn chỉ còn 10 phần trăm số hàng, kiểu cắt tỉa có cấu trúc;
4. định dạng thưa CSR của `scipy`, ở nhiều mật độ.

### Dự đoán

Cách 2 nhanh hơn cách 1 bao nhiêu lần? Và CSR phải thưa tới đâu mới thắng được ma trận dày?

In [ ]:
import scipy.sparse as sps


def do_thoi_gian(f, lap=15):
    f()                                    # một lần khởi động, không tính
    ts = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts))


dat_hat_giong()
N, B = 2048, 64
ty_le_0 = sach(90, nguon="90 phần trăm") / 100
W_day = torch.randn(N, N)
X_b = torch.randn(N, B)
W_co_0 = W_day * (torch.rand(N, N) > ty_le_0)
W_nho = W_day[: int(N * (1 - ty_le_0))].clone()        # 204 hàng: cùng số phần tử khác 0

t_day = do_thoi_gian(lambda: W_day @ X_b)
t_co_0 = do_thoi_gian(lambda: W_co_0 @ X_b)
t_nho = do_thoi_gian(lambda: W_nho @ X_b)
do_tren_may("dày 2048 nhân 2048", 1e3 * t_day, "ms")
do_tren_may("dày, 90 % là số 0", 1e3 * t_co_0, "ms")
do_tren_may("dày nhỏ hơn, 10 % số hàng", 1e3 * t_nho, "ms")
do_tren_may("tỉ số: có số 0 so với không có", t_co_0 / t_day, "lần")

X_np = X_b.numpy()
mat_do = [0.5, 0.25, 0.1, 0.05, 0.02]
t_csr = []
for d in mat_do:
    A = sps.csr_matrix((W_day * (torch.rand(N, N) < d)).numpy())
    t_csr.append(do_thoi_gian(lambda: A @ X_np))
    do_tren_may(f"CSR, mật độ {d:.0%}", 1e3 * t_csr[-1], "ms")

fig, ax = plt.subplots()
ax.plot([100 * d for d in mat_do], [1e3 * t for t in t_csr], "o-", color="C2", label="CSR (scipy)")
ax.axhline(1e3 * t_day, color="C0", label="dày, mọi mật độ")
x_ct = [100 * d for d in mat_do]
ax.plot(x_ct, [1e3 * t_day * d for d in mat_do], ls=":", color="C1",
        label="dày nhỏ hơn, nếu thời gian tỉ lệ số hàng")
ax.plot([100 * (1 - ty_le_0)], [1e3 * t_nho], "D", color="C1", ms=8, label="dày nhỏ hơn, đo được")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(x_ct, [f"{x:g}" for x in x_ct])
ax.minorticks_off()
ax.invert_xaxis()
ax.set_xlabel("mật độ: phần trăm phần tử khác 0 (thang log, thưa dần sang phải)")
ax.set_ylabel("thời gian một phép nhân (ms, thang log)")
ax.set_title("Phép nhân 2048 nhân 2048 với batch 64")
ax.legend(loc="lower left", fontsize=8)
plt.show()

Ba điều đọc được từ hình, trên một CPU thông thường:

* số 0 lưu dày không đổi thời gian: phép nhân dày vẫn nạp và nhân mọi phần tử;
* ma trận dày nhỏ hơn rẻ đi gần đúng theo số hàng nó bỏ, và đó là lý do chương nói cắt tỉa có
  cấu trúc dễ thành mức giảm độ trễ (latency) hơn;
* CSR chỉ thắng ma trận dày khi đủ thưa, và ngay cả khi thắng, nó vẫn thường thua ma trận dày nhỏ
  hơn có cùng số phần tử khác 0, vì nó phải đọc chỉ số và truy cập bộ nhớ không liên tiếp.

Điểm hoà vốn về thời gian tuỳ máy, kích thước và kernel. Chương nói không có ngưỡng độ thưa phổ
quát nào, và bạn có thể thấy điều đó bằng cách đổi `B` thành 1 rồi chạy lại.

Ví dụ 4 nhân 4 của chương cho cùng ý ở cỡ nhỏ nhất: một ma trận có 6 phần tử khác 0.

In [ ]:
M4 = torch.tensor([[1.5, 0, 0, -2.0],
                   [0, 0.7, 0, 0],
                   [3.0, 0, -1.2, 0],
                   [0, 0, 0, 0.4]])
theo_sach("phép nhân của cách tính dày", M4.numel(), sach=16, nguon="16 · 6")
theo_sach("tích khác 0 của cách tính biết độ thưa", int((M4 != 0).sum()), sach=6, nguon="16 · 6")

## 7. Số 0 có làm tệp mô hình nhỏ đi không

Một tensor dày có số 0 vẫn chiếm đúng dung lượng ban đầu. Muốn nhỏ đi, số 0 phải được **bỏ khỏi
biểu diễn**, và mỗi cách bỏ có giá của nó. Với một ma trận $r$ hàng, $n$ phần tử, mật độ $d$:

| cách lưu | byte |
|---|---|
| dày FP32 | $4n$ |
| dày cộng mặt nạ 1 bit | $4n + n/8$ |
| CSR: giá trị FP32, chỉ số cột INT32, con trỏ hàng INT32 | $8dn + 4(r+1)$ |
| COO: giá trị FP32, hai toạ độ INT32 | $12dn$ |
| có cấu trúc: ma trận dày nhỏ hơn | $4dn$ |

Chương nói với giá trị FP32 và chỉ số INT32, CSR hoà vốn quanh mật độ 50 phần trăm, chưa kể con
trỏ hàng, còn COO hoà vốn ở mật độ thấp hơn nữa.

In [ ]:
GIA_TRI = 4   # byte mỗi giá trị FP32
CHI_SO = 4    # byte mỗi chỉ số INT32


def byte_luu(r, c, d):
    n = r * c
    return {"dày FP32": GIA_TRI * n,
            "dày + mặt nạ 1 bit": GIA_TRI * n + n / 8,
            "CSR": (GIA_TRI + CHI_SO) * d * n + CHI_SO * (r + 1),
            "COO": (GIA_TRI + 2 * CHI_SO) * d * n,
            "có cấu trúc": GIA_TRI * d * n}


# Điểm hoà vốn: mật độ ở đó CSR (bỏ con trỏ hàng) bằng đúng ma trận dày, (4 + 4)·d·n = 4·n.
hoa_von_csr = GIA_TRI / (GIA_TRI + CHI_SO)
hoa_von_coo = GIA_TRI / (GIA_TRI + 2 * CHI_SO)
theo_sach("mật độ hoà vốn của CSR, chưa kể con trỏ hàng (%)", 100 * hoa_von_csr, sach=50,
          nguon="gần 50 phần trăm")
hoa_von_csr_that = (GIA_TRI * N * N - CHI_SO * (N + 1)) / ((GIA_TRI + CHI_SO) * N * N)
print(f"tính cả con trỏ hàng, ma trận {N} nhân {N}: {hoa_von_csr_that:.4%}")
print(f"COO hoà vốn ở mật độ {hoa_von_coo:.1%}")

ds = np.linspace(0.01, 1, 200)
fig, ax = plt.subplots()
for ten, mau, kieu in [("dày FP32", "C0", "-"), ("dày + mặt nạ 1 bit", "C0", ":"), ("CSR", "C2", "-"),
                       ("COO", "C3", "-"), ("có cấu trúc", "C1", "--")]:
    ax.plot(100 * ds, [byte_luu(N, N, d)[ten] / 1e6 for d in ds], kieu, color=mau, label=ten)
for d, ten, lech in [(hoa_von_csr, "CSR hoà vốn", (25, -30)), (hoa_von_coo, "COO hoà vốn", (-95, 22))]:
    ax.plot(100 * d, GIA_TRI * N * N / 1e6, "ko", ms=5)
    ax.annotate(f"{ten}: {d:.0%}", (100 * d, GIA_TRI * N * N / 1e6), xytext=lech, textcoords="offset points",
                fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xlabel("mật độ (phần trăm phần tử khác 0)")
ax.set_ylabel("dung lượng (MB)")
ax.set_title(f"Lưu một ma trận {N} nhân {N} ở FP32")
ax.set_ylim(0, 52)
ax.legend(loc="upper left", fontsize=8)
plt.show()

### Kịch bản minh hoạ của chương: từ 14 MB xuống 2 MB

Một MobileNet 14 MB phải vừa 2 MB bộ nhớ flash, và chương giả sử khoảng 85.7 phần trăm trọng số
gần bằng 0. Con số ấy đúng là tỉ lệ phải bỏ **nếu số 0 biến mất không tốn gì**, như khi cắt có
cấu trúc. Với CSR, mỗi trọng số còn lại mang thêm một chỉ số, nên phải bỏ nhiều hơn.

In [ ]:
MB_GOC = sach(14, nguon="14 MB · 2 MB")
MB_DICH = sach(2, nguon="14 MB · 2 MB")
theo_sach("tỉ lệ phải bỏ, nếu số 0 không tốn byte nào (%)", 100 * (1 - MB_DICH / MB_GOC), sach=85.7,
          nguon="85.7 phần trăm")
can_bo_csr = 1 - MB_DICH / (MB_GOC * (GIA_TRI + CHI_SO) / GIA_TRI)   # CSR: 8 byte mỗi giá trị còn lại thay vì 4
print(f"nếu lưu bằng CSR (giá trị FP32 + chỉ số INT32): phải bỏ {can_bo_csr:.1%} trọng số")

## 8. Mẫu 2 trên 4

Độ thưa có cấu trúc (structured sparsity) N trên M giữ tối đa N phần tử khác 0 trong mỗi nhóm M
phần tử liền nhau. Chương dùng 2 trên 4 làm ví dụ: phần cứng biết trước bố trí ấy, nên siêu dữ
liệu nhỏ và kernel đoán trước được. Cái giá là mất quyền chọn tự do: trong mỗi nhóm 4, phải bỏ
đúng 2, kể cả khi cả 4 đều quan trọng. Ta so nó với cắt không cấu trúc 50 phần trăm theo từng
tầng, cùng độ thưa.

In [ ]:
N_GIU, M_NHOM = sach(2, nguon="2 trên 4"), sach(4, nguon="2 trên 4")


def mat_na_n_tren_m(m, n_giu, m_nhom):
    out = []
    for l in cac_tang(m):
        w = l.weight.detach().abs()
        r, c = w.shape
        nhom = w.view(r, c // m_nhom, m_nhom)
        top = nhom.argsort(dim=2, descending=True)[:, :, :n_giu]
        out.append(torch.zeros_like(nhom).scatter_(2, top, 1.0).view(r, c))
    return out


mk24 = mat_na_n_tren_m(goc, N_GIU, M_NHOM)
for mk in mk24:
    assert mk.view(mk.shape[0], -1, M_NHOM).sum(2).eq(N_GIU).all()  # đúng 2 trong mỗi nhóm 4
do_tren_may("2 trên 4, chưa tinh chỉnh", 100 * do_chinh_xac(ap_mat_na(goc, mk24)), "%")
do_tren_may("không cấu trúc 50 % theo từng tầng", 100 * do_chinh_xac(ap_mat_na(goc, mat_na_tung_tang(goc, 0.5))), "%")

fig, axs = plt.subplots(1, 2, figsize=(7.5, 2.4))
for ax, mk, ten in [(axs[0], mat_na_tung_tang(goc, 0.5)[1], "không cấu trúc 50 %"),
                    (axs[1], mk24[1], "2 trên 4")]:
    ax.imshow(mk[:8, :32], cmap="Blues", vmin=0, vmax=1.3)
    ax.set_title(f"{ten}: 8 hàng, 32 cột đầu", fontsize=9)
    ax.set_xticks(np.arange(-0.5, 32, 4), [])
    ax.set_yticks([])
    ax.grid(True, axis="x", color="0.3", lw=0.6, alpha=1)
plt.tight_layout()
plt.show()

Ô sẫm là trọng số được giữ. Bên phải, mỗi khung 4 cột có đúng 2 ô sẫm trên mỗi hàng. Bên trái,
số ô giữ lại trong một khung thay đổi tuỳ ý, và đó là thứ phần cứng không đoán trước được.

## Thử thêm

1. Ở phần 6, đổi `B = 64` thành `B = 1` (một phép nhân ma trận với vector) rồi chạy lại ô ấy.
   Điểm hoà vốn về thời gian của CSR dịch về phía nào?
2. Ở phần 4, cắt 98 phần trăm thay vì 95 rồi tinh chỉnh 1, 2, 3 epoch. Từ đâu thì tinh chỉnh
   không kéo độ chính xác về được nữa?
3. Ở phần 7, đặt `CHI_SO = 2` (chỉ số INT16, đủ cho một ma trận 2048 cột). CSR bây giờ hoà vốn
   ở mật độ bao nhiêu, và kịch bản 14 MB phải bỏ bao nhiêu? (Dấu ✗ khi đó là điều được chờ
   đợi: sách tính với INT32.)

## Tóm lại

* Cắt tỉa theo độ lớn là một quy tắc kinh nghiệm cho bài toán tổ hợp $\|\hat{\mathbf W}\|_0 \le k$,
  và cắt toàn cục để thang đo của từng tầng tự quyết tầng nào mất nhiều hơn.
* Ở cùng số trọng số, cắt không cấu trúc thường giữ độ chính xác tốt hơn, còn cắt có cấu trúc
  cho ma trận dày nhỏ hơn mà kernel thông thường chạy nhanh.
* Số 0 lưu dày không đổi thời gian lẫn dung lượng. Định dạng thưa chỉ có lợi sau điểm hoà vốn,
  khoảng 50 phần trăm mật độ cho CSR về byte, và thấp hơn nhiều về thời gian trên CPU.
* Mẫu 2 trên 4 đổi một chút tự do chọn lấy một bố trí mà phần cứng khai thác được.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Nén mô hình — đổi phần dư thừa lấy ràng buộc triển khai](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch10-model-compression/), dựng từ chương
[*Model Compression*](https://mlsysbook.ai/vol1/model_compression/model_compression.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_10_model_compress.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.